# Notebook 04 — RAG Pipeline + Full Results Aggregation
**AITR_Lab02_RushikeshShirsath_LLMFactuality**

This notebook:
1. Verifies the BM25 RAG pipeline in detail
2. Aggregates all results CSVs into a single `all_results.csv` summary table
3. Generates Table 1 (ready to paste into LaTeX)
4. Runs `plot_results.py` to generate all three figures

**Pre-condition:** Notebooks 01, 02, 03 must be complete for all 4 conditions.

In [ ]:
# Cell 1: Setup
import sys, os, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/content')
os.makedirs('results/figures', exist_ok=True)
print('Setup OK')

In [ ]:
# Cell 2: Verify BM25 RAG pipeline quality
from src.rag import build_bm25_index, retrieve

print('Building BM25 index...')
bm25, doc_texts = build_bm25_index(n_test=500)
print(f'Index: {len(doc_texts)} docs')

# Manual spot-check of 5 queries
spot_queries = [
    'Does metformin reduce cardiovascular risk in type 2 diabetes?',
    'Is aspirin effective for primary prevention of stroke?',
    'Does exercise improve outcomes in heart failure patients?',
    'Can statins reduce mortality in elderly patients?',
    'Does antibiotic prophylaxis prevent surgical site infection?',
]
print('\n=== BM25 Retrieval Spot-Check ===')
for q in spot_queries:
    result = retrieve(bm25, doc_texts, q, top_k=1)
    print(f'\nQ: {q}')
    print(f'A: {result[:200]}...')
print('\n>>> Manually assess whether each retrieved abstract is topically relevant.')
print('>>> Expected: at least 4/5 relevant (8/10 verified in ai_assistance_log.md)')

In [ ]:
# Cell 3: Aggregate all summary JSONs into one table
summary_files = glob.glob('results/summary_*.json')
print(f'Found {len(summary_files)} summary files: {summary_files}')

rows = []
for f in summary_files:
    with open(f) as fp:
        rows.append(json.load(fp))

if rows:
    df_summary = pd.DataFrame(rows)
    print('\nAll results:')
    print(df_summary.to_string(index=False))
    df_summary.to_csv('results/all_results.csv', index=False)
    print('\nSaved: results/all_results.csv')
else:
    print('No summary files yet. Run notebooks 02 and 03 for all conditions first.')

In [ ]:
# Cell 4: Generate Table 1 (LaTeX-ready)
print('\n=== TABLE 1 (LaTeX format — paste into paper/main.tex) ===')
print('Replace [RUN] values with the numbers below after all runs complete.\n')

# Majority-class row
print(f'Majority-class & No  & 55.0\\% & --- & 0.000 & --- \\\\')

if rows:
    mc_rows = [r for r in rows if 'accuracy' in r]        # MC summaries
    oe_rows = [r for r in rows if 'mean_token_f1' in r]   # OE summaries

    for model in ['llama', 'mistral']:
        for rag_tag in ['norag', 'rag']:
            rag_label = 'No' if rag_tag == 'norag' else 'BM25'
            mc = next((r for r in mc_rows if r.get('model')==model and r.get('rag')==rag_tag), {})
            oe = next((r for r in oe_rows if r.get('model')==model and r.get('rag')==rag_tag), {})

            mc_acc = f"{mc.get('accuracy',0)*100:.1f}\\%" if mc else '[RUN]'
            mc_f1  = f"{mc.get('macro_f1',0):.4f}"        if mc else '[RUN]'
            oe_f1  = f"{oe.get('mean_token_f1',0):.4f}"   if oe else '[RUN]'
            gap    = f"{(mc.get('accuracy',0) - oe.get('mean_token_f1',0))*100:.1f}" if (mc and oe) else '[RUN]'

            model_label = 'Llama-3.1-8B' if model == 'llama' else 'Mistral-7B'
            print(f'{model_label} & {rag_label} & {mc_acc} & {mc_f1} & {oe_f1} & {gap} \\\\')
else:
    for model_label in ['Llama-3.1-8B', 'Mistral-7B']:
        for rag_label in ['No', 'BM25']:
            print(f'{model_label} & {rag_label} & [RUN] & [RUN] & [RUN] & [RUN] \\\\')

In [ ]:
# Cell 5: Run plot_results.py to generate all three figures
!python scripts/plot_results.py --input results/ --output results/figures/
print('\nFigures saved to results/figures/')

In [ ]:
# Cell 6: Display all three figures inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_files = [
    ('results/figures/figure1_bar_mc_vs_oe.png',     'Figure 1: MC vs. Open-Ended by Condition'),
    ('results/figures/figure2_confusion_matrices.png','Figure 2: Per-Class Confusion Matrices'),
    ('results/figures/figure3_f1_distributions.png', 'Figure 3: Token F1 Distributions'),
]

for path, title in fig_files:
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(12, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Figure not yet generated: {path}')

In [ ]:
# Cell 7: Final submission checklist
checklist = [
    ('results/mc_results_llama_norag.csv',         'Llama no-RAG MC results'),
    ('results/mc_results_llama_rag.csv',           'Llama BM25 MC results'),
    ('results/mc_results_mistral_norag.csv',       'Mistral no-RAG MC results'),
    ('results/mc_results_mistral_rag.csv',         'Mistral BM25 MC results'),
    ('results/openended_results_llama_norag.csv',  'Llama no-RAG OE results'),
    ('results/openended_results_llama_rag.csv',    'Llama BM25 OE results'),
    ('results/openended_results_mistral_norag.csv','Mistral no-RAG OE results'),
    ('results/openended_results_mistral_rag.csv',  'Mistral BM25 OE results'),
    ('results/all_results.csv',                    'Aggregated summary table'),
    ('results/figures/figure1_bar_mc_vs_oe.png',   'Figure 1'),
    ('results/figures/figure2_confusion_matrices.png', 'Figure 2'),
    ('results/figures/figure3_f1_distributions.png',   'Figure 3'),
    ('paper/main.tex',                             'LaTeX manuscript'),
    ('paper/references.bib',                       'BibTeX references'),
    ('research_decisions.md',                      'Decision log'),
    ('baseline_validation.md',                     'Validation report'),
    ('ai_assistance_log.md',                       'AI assistance log'),
    ('README.md',                                  'README'),
    ('requirements.txt',                           'Dependencies'),
]
print('\n=== SUBMISSION CHECKLIST ===')
all_ok = True
for path, label in checklist:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {label}')
    if not exists: all_ok = False

print()
if all_ok:
    print('  ALL FILES PRESENT — ready to submit!')
else:
    print('  Some files missing — complete all experiment runs first.')